# Geolytica vs UKMap — rooftop address match analysis

Reads two GBR address-point File Geodatabases — **Geolytica** and **UKMap** — and finds the records present in both.

* **Match key:** Geolytica `HOUSE_NUMBER_N` + `ADDRESS_N` + `POSTAL_CODE_N` ⇔ UKMap `HouseNumber_N` + `ADDRESS_N` + `PostalCode_N` (exact; a null in any key never matches), **and** the two points must be within a 250 m buffer of each other — key matches further apart are dropped as different addresses.
* **Stats:** record counts per source, distinct keys, common keys, matched records and joined pairs.
* **BFP check:** each point is tested against MCR building footprints (`polygons_nexventura`, GBR, `building` set and not `no`) on the central MCR warehouse — needs TomTom VPN and `DATABRICKS_TOKEN`.
* **Output:** one row per matched pair — `housenumber_n`, `address_n`, `postcode_n`, `location_geolytic`, `location_ukmap` (WKT), `distance_between_both` (metres, haversine, 2 dp), `geolytic_inside_bfp` / `ukmap_inside_bfp` — written to one `.gdb` as three layers: `location_ukmap` (UKMap point), `location_geolytics` (Geolytica point) and `linestring_between_ukmap_geolytic` (line joining the two).

All layers of each `.gdb` are read (UKMap has `AS_91`, `AS_92`, `AS_93`). Postcode keeps same-named streets in different towns apart; a few keys still repeat within a source (e.g. flats sharing a house number), so those join many-to-many.

In [26]:
# Read every layer of the Geolytica gdb into one GeoDataFrame (keys + point geometry only).
%pip install -q geopandas pyogrio shapely numpy pandas databricks-sql-connector
import os
import shutil

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import shapely

GEOLYTICA_GDB = "/Users/sawan.darekar/Desktop/workspace_data/gbr-regression/Geolytica-AmXBq0lR_ED0NTbCjMlr6.gdb"
UKMAP_GDB = "/Users/sawan.darekar/Desktop/workspace_data/gbr-regression/UKMap-LsecDHSS6zm6-kwslTf9r.gdb"
OUTPUT_GDB = "/Users/sawan.darekar/Desktop/workspace_data/gbr-regression/geolytic_ukmap_common.gdb"
MATCH_BUFFER_M = 250  # key matches whose two points are further apart than this are not the same address

def read_all_layers(gdb_path, columns):
    frames = [
        pyogrio.read_dataframe(gdb_path, layer=layer, columns=columns).assign(source_layer=layer)
        for layer, _ in pyogrio.list_layers(gdb_path)
    ]
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=frames[0].crs)

geolytica = read_all_layers(GEOLYTICA_GDB, ["HOUSE_NUMBER_N", "ADDRESS_N", "POSTAL_CODE_N"])
geolytica.head()


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


,ADDRESS_N,HOUSE_NUMBER_N,POSTAL_CODE_N,geometry,source_layer
0,Bridgeness Lane,14,EH51 9LQ,POINT (-3.58233 56.01623),AS_91
1,Park Circle,1,KY7 6AU,POINT (-3.13849 56.20087),AS_91
2,Edward Place,10,DD2 5QH,POINT (-3.06659 56.49782),AS_91
3,Whitelaw Crescent,45,KY11 4RR,POINT (-3.43301 56.06467),AS_91
4,Stirling Terrace,17,DD3 6JX,POINT (-2.98143 56.46851),AS_91


In [27]:
# Read every layer of the UKMap gdb into one GeoDataFrame (keys + point geometry only).
ukmap = read_all_layers(UKMAP_GDB, ["HouseNumber_N", "ADDRESS_N", "PostalCode_N"])
ukmap.head()

,ADDRESS_N,HouseNumber_N,PostalCode_N,geometry,source_layer
0,Low Road,5,DL2 3DW,POINT (-1.73892 54.54652),AS_91
1,Patten Lane,4,TS14 6PJ,POINT (-1.05095 54.53655),AS_91
2,Bell Avenue,77,DH6 5PE,POINT (-1.53399 54.74115),AS_91
3,Lindsay Circus,39,EH24 9EN,POINT (-3.14572 55.84946),AS_91
4,Ormiston Road,7,EH33 2DS,POINT (-2.94937 55.94291),AS_91


In [28]:
# Inner-join both sources on house number + street name + postcode, keeping only pairs whose points are within MATCH_BUFFER_M of each other.
def keyed_points(gdf, house_number_col, postcode_col):
    return pd.DataFrame({
        "rid": gdf.index,
        "housenumber_n": gdf[house_number_col],
        "address_n": gdf["ADDRESS_N"],
        "postcode_n": gdf[postcode_col],
        "lon": gdf.geometry.x,
        "lat": gdf.geometry.y,
    }).dropna(subset=["housenumber_n", "address_n", "postcode_n"])

geo_keys = keyed_points(geolytica, "HOUSE_NUMBER_N", "POSTAL_CODE_N")
uk_keys = keyed_points(ukmap, "HouseNumber_N", "PostalCode_N")

joined = geo_keys.merge(uk_keys, on=["housenumber_n", "address_n", "postcode_n"], suffixes=("_geolytic", "_ukmap"))

lat1, lon1, lat2, lon2 = map(np.radians, [joined.lat_geolytic, joined.lon_geolytic, joined.lat_ukmap, joined.lon_ukmap])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
joined["distance_between_both"] = (2 * 6_371_008.8 * np.arcsin(np.sqrt(a))).round(2)  # metres, 2 dp
key_matched_pairs = len(joined)
joined = joined[joined.distance_between_both <= MATCH_BUFFER_M].reset_index(drop=True)

geo_points = shapely.points(joined.lon_geolytic, joined.lat_geolytic)
joined["location_geolytic"] = shapely.to_wkt(geo_points, rounding_precision=7)
joined["location_ukmap"] = shapely.to_wkt(shapely.points(joined.lon_ukmap, joined.lat_ukmap), rounding_precision=7)

result = gpd.GeoDataFrame(
    joined[["housenumber_n", "address_n", "postcode_n", "location_geolytic", "location_ukmap", "distance_between_both"]],
    geometry=geo_points,
    crs=geolytica.crs,
)
result.head()

,housenumber_n,address_n,postcode_n,location_geolytic,location_ukmap,distance_between_both,geometry
0,37,Bellerophon Drive,EH26 8NU,POINT (-3.21798 55.8262),POINT (-3.2181504 55.826236),11.37,POINT (-3.21798 55.8262)
1,33,Strathyre Drive,EH47 8AZ,POINT (-3.62945 55.8487),POINT (-3.629154 55.8488131),22.35,POINT (-3.62945 55.8487)
2,19,Park Street,FK12 5LW,POINT (-3.80176 56.15362),POINT (-3.8015424 56.1535766),14.31,POINT (-3.80176 56.15362)
3,123,Arthur View Crescent,EH22 1QS,POINT (-3.11258 55.91615),POINT (-3.1125924 55.9160533),10.78,POINT (-3.11258 55.91615)
4,62,Walker Drive,EH30 9RR,POINT (-3.40913 55.99056),POINT (-3.4094914 55.9905639),22.48,POINT (-3.40913 55.99056)


In [29]:
# Flag whether each Geolytica / UKMap point lies inside an MCR building footprint (point-in-polygon runs on the central MCR warehouse).
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

from databricks import sql

MCR_HOST = "adb-879908127091742.2.azuredatabricks.net"
MCR_HTTP_PATH = "/sql/1.0/warehouses/dad35031bafe9507"
MCR_BFP_TABLE = "pu_orbis_platform_prod_catalog.map_central_repository.polygons_nexventura_26430_000"
CHUNK_SIZE = 250_000  # ~140 s per query (~7.6 MB of SQL), well under the 10-min MCR query timeout and 16 MB statement limit
PARALLEL_QUERIES = 2  # MCR policy: max 2 queries in parallel per user

# Fail fast with the server's real reason (bad token / VPN) instead of the connector's silent Thrift retries.
if not os.environ.get("DATABRICKS_TOKEN"):
    raise RuntimeError("DATABRICKS_TOKEN is not set in this kernel - export it in ~/.zshrc and restart the IDE/kernel")
try:
    urllib.request.urlopen(urllib.request.Request(
        f"https://{MCR_HOST}/api/2.0/preview/scim/v2/Me",
        headers={"Authorization": f"Bearer {databricks_token}"}), timeout=30)
except urllib.error.HTTPError as err:
    raise RuntimeError(f"MCR rejected the connection (HTTP {err.code}): {err.read().decode()[:300]}") from None
except urllib.error.URLError as err:
    raise RuntimeError(f"Cannot reach MCR - check the TomTom VPN (Global Connect): {err.reason}") from None

coords = np.round(np.concatenate([
    joined[["lon_geolytic", "lat_geolytic"]].to_numpy(),
    joined[["lon_ukmap", "lat_ukmap"]].to_numpy(),
]), 7)
unique_coords, inverse = np.unique(coords, axis=0, return_inverse=True)

def points_inside_bfp(start):
    chunk = unique_coords[start:start + CHUNK_SIZE]
    values = ",".join(f"({start + i},{lon:.7f},{lat:.7f})" for i, (lon, lat) in enumerate(chunk))
    # Each building is parsed once and its bbox computed; the bbox range check cuts ~765 h3 candidates per point
    # down to a few before the exact ST_Contains (parsing WKT per candidate pair made the query ~3x slower).
    query = f"""
        WITH pts AS (SELECT * FROM VALUES {values} AS t(pid, lon, lat)),
        cells AS (
            SELECT pid, lon, lat, h3_toparent(h3_longlatash3string(lon, lat, 8), r) AS cell
            FROM pts LATERAL VIEW explode(sequence(2, 8)) AS r
        ),
        bld AS (
            SELECT h3_index, g, ST_XMin(g) AS xmin, ST_XMax(g) AS xmax, ST_YMin(g) AS ymin, ST_YMax(g) AS ymax
            FROM (
                SELECT h3_index, ST_GeomFromWKT(geometry) AS g
                FROM {MCR_BFP_TABLE}
                WHERE license_zone LIKE '%GBR%'
                  AND building IS NOT NULL AND building <> 'no'
                  AND h3_index IN (SELECT cell FROM cells)
            )
        )
        SELECT DISTINCT c.pid
        FROM cells c
        JOIN bld b
          ON b.h3_index = c.cell
         AND c.lon BETWEEN b.xmin AND b.xmax
         AND c.lat BETWEEN b.ymin AND b.ymax
        WHERE ST_Contains(b.g, ST_Point(c.lon, c.lat))
    """
    with sql.connect(server_hostname=MCR_HOST, http_path=MCR_HTTP_PATH, access_token="{databricks_token}",
                     user_agent_entry="geolytic-ukmap-rooftop-analysis") as conn, conn.cursor() as cursor:
        cursor.execute(query)
        inside_ids = [row[0] for row in cursor.fetchall()]
    print(f"chunk {start:,}-{start + len(chunk):,}: {len(inside_ids):,} inside")
    return inside_ids

with ThreadPoolExecutor(max_workers=PARALLEL_QUERIES) as pool:
    inside_ids = [pid for ids in pool.map(points_inside_bfp, range(0, len(unique_coords), CHUNK_SIZE)) for pid in ids]

inside = np.zeros(len(unique_coords), dtype=bool)
inside[inside_ids] = True
flags = inside[inverse.ravel()]
result["geolytic_inside_bfp"] = flags[:len(joined)]
result["ukmap_inside_bfp"] = flags[len(joined):]
result.head()

chunk 0-250,000: 153,366 inside
chunk 250,000-500,000: 155,541 inside
chunk 750,000-901,766: 101,839 inside
chunk 500,000-750,000: 173,182 inside


,housenumber_n,address_n,postcode_n,location_geolytic,location_ukmap,distance_between_both,geometry,geolytic_inside_bfp,ukmap_inside_bfp
0,37,Bellerophon Drive,EH26 8NU,POINT (-3.21798 55.8262),POINT (-3.2181504 55.826236),11.37,POINT (-3.21798 55.8262),True,True
1,33,Strathyre Drive,EH47 8AZ,POINT (-3.62945 55.8487),POINT (-3.629154 55.8488131),22.35,POINT (-3.62945 55.8487),True,True
2,19,Park Street,FK12 5LW,POINT (-3.80176 56.15362),POINT (-3.8015424 56.1535766),14.31,POINT (-3.80176 56.15362),True,True
3,123,Arthur View Crescent,EH22 1QS,POINT (-3.11258 55.91615),POINT (-3.1125924 55.9160533),10.78,POINT (-3.11258 55.91615),True,True
4,62,Walker Drive,EH30 9RR,POINT (-3.40913 55.99056),POINT (-3.4094914 55.9905639),22.48,POINT (-3.40913 55.99056),True,True


In [30]:
# Compute record counts per source, distinct/common keys, matched records and joined pairs (after the MATCH_BUFFER_M filter).
key = ["housenumber_n", "address_n", "postcode_n"]
geo_distinct = geo_keys[key].drop_duplicates()
uk_distinct = uk_keys[key].drop_duplicates()

stats = pd.Series({
    "geolytica_records": len(geolytica),
    "ukmap_records": len(ukmap),
    "geolytica_distinct_keys": len(geo_distinct),
    "ukmap_distinct_keys": len(uk_distinct),
    "key_matched_pairs": key_matched_pairs,
    f"pairs_dropped_beyond_{MATCH_BUFFER_M}m": key_matched_pairs - len(result),
    "common_distinct_keys": len(joined[key].drop_duplicates()),
    "geolytica_records_matched": joined.rid_geolytic.nunique(),
    "ukmap_records_matched": joined.rid_ukmap.nunique(),
    "joined_pairs": len(result),
    "pairs_within_10m": (result.distance_between_both <= 10).sum(),
    "pairs_within_50m": (result.distance_between_both <= 50).sum(),
    "median_distance_m": round(result.distance_between_both.median(), 2),
    "geolytic_inside_bfp": result.geolytic_inside_bfp.sum(),
    "ukmap_inside_bfp": result.ukmap_inside_bfp.sum(),
    "both_inside_bfp": (result.geolytic_inside_bfp & result.ukmap_inside_bfp).sum(),
}, dtype=object)
ukmap_by_layer = ukmap.source_layer.value_counts().sort_index()

In [31]:
# Print the summary stats.
print(stats.to_string())
print("\nUKMap records per layer:")
print(ukmap_by_layer.to_string())

geolytica_records            3624617
ukmap_records                3636491
geolytica_distinct_keys      3601715
ukmap_distinct_keys          3433973
key_matched_pairs             586650
pairs_dropped_beyond_250m        583
common_distinct_keys          558143
geolytica_records_matched     562286
ukmap_records_matched         581763
joined_pairs                  586067
pairs_within_10m               54359
pairs_within_50m              554067
median_distance_m              17.84
geolytic_inside_bfp           286421
ukmap_inside_bfp              532915
both_inside_bfp               268620

UKMap records per layer:
source_layer
AS_91    3010398
AS_92     598694
AS_93      27399


In [32]:
# Write the joined pairs to a new File Geodatabase as three layers (UKMap point, Geolytica point, line between them), replacing any previous run.
if os.path.exists(OUTPUT_GDB):
    shutil.rmtree(OUTPUT_GDB)

ukmap_xy = joined[["lon_ukmap", "lat_ukmap"]].to_numpy()
geolytic_xy = joined[["lon_geolytic", "lat_geolytic"]].to_numpy()
attributes = pd.DataFrame(result.drop(columns="geometry"))
layers = {
    "location_ukmap": shapely.points(ukmap_xy),
    "location_geolytics": shapely.points(geolytic_xy),
    "linestring_between_ukmap_geolytic": shapely.linestrings(np.stack([ukmap_xy, geolytic_xy], axis=1)),
}
for layer, geometry in layers.items():
    pyogrio.write_dataframe(gpd.GeoDataFrame(attributes, geometry=geometry, crs=result.crs), OUTPUT_GDB, layer=layer, driver="OpenFileGDB")
    print(f"Wrote {len(attributes):,} rows to {OUTPUT_GDB} layer {layer}")

Wrote 586,067 rows to /Users/sawan.darekar/Desktop/workspace_data/gbr-regression/geolytic_ukmap_common.gdb layer location_ukmap
Wrote 586,067 rows to /Users/sawan.darekar/Desktop/workspace_data/gbr-regression/geolytic_ukmap_common.gdb layer location_geolytics
Wrote 586,067 rows to /Users/sawan.darekar/Desktop/workspace_data/gbr-regression/geolytic_ukmap_common.gdb layer linestring_between_ukmap_geolytic


In [33]:
# Read OUTPUT_GDB back and compare how often each source's point lies inside a building footprint (higher = better rooftop placement).
out = pyogrio.read_dataframe(OUTPUT_GDB, layer="location_ukmap", read_geometry=False)
geo_in = out.geolytic_inside_bfp.astype(bool)
uk_in = out.ukmap_inside_bfp.astype(bool)
total = len(out)

categories = {
    "both inside BFP": geo_in & uk_in,
    "only Geolytica inside": geo_in & ~uk_in,
    "only UKMap inside": ~geo_in & uk_in,
    "neither inside": ~geo_in & ~uk_in,
}
breakdown = pd.DataFrame({
    "pairs": {name: mask.sum() for name, mask in categories.items()},
    "pct": {name: f"{mask.mean():.1%}" for name, mask in categories.items()},
    "median_distance_m": {name: round(out.distance_between_both[mask].median(), 1) for name, mask in categories.items()},
})

geo_pct, uk_pct = geo_in.mean(), uk_in.mean()
only_geo, only_uk = categories["only Geolytica inside"].sum(), categories["only UKMap inside"].sum()
better, worse = ("UKMap", "Geolytica") if uk_pct > geo_pct else ("Geolytica", "UKMap")

print(f"Exact matching pairs (house number + street + postcode, within {MATCH_BUFFER_M} m): {total:,}")
print(f"Distinct matching addresses: {len(out[['housenumber_n', 'address_n', 'postcode_n']].drop_duplicates()):,}")
print(f"Geolytica inside BFP: {geo_in.sum():,} ({geo_pct:.1%})")
print(f"UKMap inside BFP:     {uk_in.sum():,} ({uk_pct:.1%})")
print()
print(breakdown.to_string())
print()
print(f"Verdict: {better} is better - its point is inside a building for {max(geo_pct, uk_pct):.1%} of matched addresses "
      f"vs {min(geo_pct, uk_pct):.1%} for {worse} ({abs(uk_pct - geo_pct) * 100:.1f} pp higher). "
      f"When the sources disagree, {better} is the one inside the building in "
      f"{max(only_geo, only_uk) / (only_geo + only_uk):.1%} of cases ({max(only_geo, only_uk):,} vs {min(only_geo, only_uk):,}).")

Exact matching pairs (house number + street + postcode, within 250 m): 586,067
Distinct matching addresses: 558,143
Geolytica inside BFP: 286,421 (48.9%)
UKMap inside BFP:     532,915 (90.9%)

                        pairs    pct  median_distance_m
both inside BFP        268620  45.8%               15.3
only Geolytica inside   17801   3.0%               17.8
only UKMap inside      264295  45.1%               19.4
neither inside          35351   6.0%               18.2

Verdict: UKMap is better - its point is inside a building for 90.9% of matched addresses vs 48.9% for Geolytica (42.1 pp higher). When the sources disagree, UKMap is the one inside the building in 93.7% of cases (264,295 vs 17,801).
